# 🏭 Урок 10 — Pipeline и кросс-валидация: конвейер для ML

**Что мы сделаем сегодня:**
1. Соберём **Pipeline** — конвейер, который сам готовит данные и обучает модель.
2. Научим его обрабатывать числовые и категориальные признаки по-разному (`ColumnTransformer`).
3. Оценим модель **честно** через кросс-валидацию (среднее ± разброс).
4. Устроим соревнование класса: у кого выше точность.

> 🏭 **Аналогия:** Pipeline — это конвейер на кухне. Сырые продукты входят с одного конца, готовое блюдо выходит с другого. Шаги всегда в правильном порядке — ничего не забудешь.

> ★ **С этого урока Pipeline — обязательный стандарт курса.** Все ваши проекты дальше собираются как Pipeline.

## Блок 1 · Данные КАК ЕСТЬ — ничего не чистим руками

**Что делаем:** берём Titanic и оставляем пропуски и текстовые категории **как есть**.

**Зачем:** всю подготовку сделает Pipeline. Наша задача — только указать, где числа, а где категории.

**Что ожидаем:** таблицу с пропусками и текстом — и это нормально.

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split

df = sns.load_dataset('titanic')

# Разделяем признаки по типу — обрабатывать будем по-разному
num_features = ['age', 'fare', 'sibsp', 'parch']   # числа
cat_features = ['sex', 'pclass', 'embarked']         # категории

X = df[num_features + cat_features]   # НЕ чистим вручную!
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
X.head()

## Блок 2 · ColumnTransformer: своя обработка для чисел и категорий

**Что делаем:** описываем два мини-конвейера.
- **Числа:** заполнить пропуски медианой → масштабировать.
- **Категории:** заполнить самым частым значением → one-hot (превратить в столбцы 0/1).

**Зачем:** нельзя масштабировать слово 'female' и нельзя one-hot-кодировать возраст. Разным типам — разная обработка.

**Что ожидаем:** пока ничего не печатается — мы только описываем шаги.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Числа: заполнить медианой -> масштабировать
num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])

# Категории: заполнить самым частым -> one-hot (0/1 столбцы)
cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

# Собираем: какому столбцу какой мини-конвейер
preprocess = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features),
])
print('Препроцессор собран ✅')

## Блок 3 · Полный Pipeline: подготовка + модель в одном объекте

**Что делаем:** соединяем препроцессор и модель в единый Pipeline.

**Зачем — самое важное:** `.fit()` считает медиану и масштаб **только по train**. Тест остаётся нетронутым — нет **утечки данных**.

**Что ожидаем:** точность на test, и ни одной ручной строки `fillna`/`get_dummies`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

model = Pipeline([
    ('prep', preprocess),
    ('forest', RandomForestClassifier(n_estimators=100, random_state=42)),
])

# Одна строка .fit делает ВСЕ шаги по порядку, сам
model.fit(X_train, y_train)
acc = accuracy_score(y_test, model.predict(X_test))
print(f'Одна проверка (test): {acc:.1%}')

**Вывод:** весь код подготовки исчез внутрь конвейера. Модель принимает **сырые** данные и сама делает всё по порядку. Это и есть защита от утечки.

## Блок 4 · Честная оценка: кросс-валидация

**Что делаем:** проверяем модель не один раз, а 5 раз на разных частях данных.

**Зачем:** одной проверке могло «повезти» с удачным разбиением. Пять проверок честнее.

**Что ожидаем:** среднее и **разброс** (±). Большой разброс = модели нельзя слепо доверять.

> Аналогия: оценить ученика по одной контрольной или по пяти? Пять — надёжнее.

In [ ]:
from sklearn.model_selection import cross_val_score

# cv=5 -> данные делятся на 5 частей, 5 раз обучаем и проверяем
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')

print('Точность на 5 фолдах:', [f'{s:.1%}' for s in scores])
print(f'Среднее: {scores.mean():.1%}  ±  {scores.std():.1%}')

## Блок 5 (соревнование) 🏆 · Подними точность!

**Что делаем:** меняем **только последний шаг** конвейера — препроцессинг переиспользуем.

**Зачем:** увидеть, как легко экспериментировать, когда подготовка уже автоматизирована.

**Идеи:** число деревьев, `max_depth`, другая модель, добавить/убрать признаки.

👉 Запиши свой результат в лидерборд на доске!

In [ ]:
# Пример улучшения — меняем параметры леса
model_v2 = Pipeline([
    ('prep', preprocess),
    ('forest', RandomForestClassifier(
        n_estimators=300, max_depth=6, random_state=42)),
])

scores = cross_val_score(model_v2, X, y, cv=5)
print(f'Мой результат: {scores.mean():.1%} ± {scores.std():.1%}')

---
## 🏠 Домашнее задание
1. Собери Pipeline (`ColumnTransformer` + модель) на своём датасете.
2. Выведи `cross_val_score` как «среднее ± разброс» (cv=5).
3. Письменно (3–4 предложения): зачем нужен `ColumnTransformer` и как Pipeline защищает от утечки данных.

**Сдача:** ссылка на ноутбук в Telegram. С этого урока проверяю: **вся подготовка данных — внутри Pipeline.**

---
## ⭐ Для сильных учеников

### A. GridSearchCV прямо поверх Pipeline (подбор + проверка без утечки)

In [ ]:
from sklearn.model_selection import GridSearchCV

# К параметрам шага обращаемся через имя_шага__параметр
grid = {
    'forest__n_estimators': [100, 300],
    'forest__max_depth': [4, 6, None],
}
search = GridSearchCV(model, grid, cv=5, scoring='accuracy')
search.fit(X, y)
print('Лучшие параметры:', search.best_params_)
print(f'Лучшая точность (CV): {search.best_score_:.1%}')

### B. Свой признак через FunctionTransformer
Добавим признак «размер семьи» = sibsp + parch + 1 (сам пассажир).

In [ ]:
import numpy as np
from sklearn.preprocessing import FunctionTransformer

def add_family(df_in):
    df_out = df_in.copy()
    df_out['family'] = df_out['sibsp'] + df_out['parch'] + 1
    return df_out

# Пробуем на копии данных
ft = FunctionTransformer(add_family)
print(ft.transform(X).head())

### C. Регрессионный вариант «House Prices» (связь с уроком 8 — MAE)
Тот же принцип Pipeline, но модель — **регрессор** (предсказываем число — цену), а метрика — средняя ошибка **MAE**. Датасет цен создаём сами, чтобы код работал без загрузок.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# Небольшой датасет о квартирах (площадь, комнаты, этаж, возраст дома)
rng = np.random.default_rng(42)
n = 400
area  = rng.integers(30, 120, n)      # площадь, м2
rooms = rng.integers(1, 5, n)          # число комнат
floor = rng.integers(1, 20, n)         # этаж
age   = rng.integers(0, 50, n)         # возраст дома, лет
# Цена (тыс. $) зависит от признаков + случайный шум
price = 20 + area*1.5 + rooms*8 - age*0.6 + rng.normal(0, 12, n)

houses = pd.DataFrame({'area': area, 'rooms': rooms, 'floor': floor, 'age': age})
target = price

reg = Pipeline([
    ('scale', StandardScaler()),
    ('forest', RandomForestRegressor(n_estimators=100, random_state=42)),
])

# Для регрессии оцениваем MAE (средняя ошибка; чем меньше — тем лучше)
mae = -cross_val_score(reg, houses, target, cv=5,
                       scoring='neg_mean_absolute_error')
print(f'Средняя ошибка MAE: {mae.mean():.1f} ± {mae.std():.1f} тыс. $')
# То есть в среднем модель промахивается по цене примерно на эту сумму